# ДЗ 7. Часові ряди та напівструктуровані дані у задачах ML

**Курс:** Реляційні бази даних · **Тема 7** · репозиторій `goit-rdb-hw-07`

Pipeline у цьому notebook:

1. **Setup** — `pgserver` (PostgreSQL усередині Colab) + завантаження NYC TLC Yellow Taxi (fallback — stub-schema).
2. **Time-series ML features** — lag / rolling / expanding ознаки на `t7_hw_taxi_daily` (усі вікна з `PARTITION BY pickup_zone_id`).
3. **JSONB event-log** `t7_hw_events` — 5 типів подій із різними схемами payload; `@>`, `?`, `?|`, JSONPath.
4. **Індекси** — GIN (`jsonb_path_ops`) та expression B-tree, виміряні через `EXPLAIN (ANALYZE, BUFFERS)`.
5. **JSONB vs normalized** + bonus MongoDB pseudocode + **Reflection**.

> Notebook розрахований на **Runtime → Restart & Run All**: усі toy-таблиці створюються через `DROP ... IF EXISTS`, дані завантажуються кодом, жодних ручних кроків.

## Завдання 1. Setup і завантаження даних

### 1.1. Залежності, запуск pgserver, перевірка підключення

In [ ]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests

In [ ]:
import os
import re
from urllib.request import urlretrieve

import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

# У Colab піднімаємо PostgreSQL через pgserver.
# (HW7_PG_URI — лише запасний шлях для локального запуску з уже наявним PostgreSQL.)
PG_URI = os.environ.get('HW7_PG_URI')
if PG_URI is None:
    import pgserver
    pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
    PG_URI = pg.get_uri()

engine = create_engine(PG_URI, future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

### 1.2. Допоміжні функції

* `run_sql(script)` — виконує DDL/DML (можна кілька statements) в одній транзакції;
* `q(sql)` — повертає результат `SELECT` як `DataFrame`;
* `explain(label, sql)` — друкує план `EXPLAIN (ANALYZE, BUFFERS)` і зберігає його в `EXPLAINS[label]`, щоб наприкінці зібрати таблицю метрик для Reflection.

SQL передається драйверу «як є» (`exec_driver_sql`), тому оператор `%` у запитах не потребує екранування.

In [ ]:
EXPLAINS = {}


def run_sql(script: str) -> None:
    """DDL / DML: один або кілька statements в одній транзакції."""
    with engine.begin() as conn:
        conn.exec_driver_sql(script)


def q(sql: str) -> pd.DataFrame:
    """SELECT -> DataFrame."""
    with engine.connect() as conn:
        res = conn.exec_driver_sql(sql)
        return pd.DataFrame(res.fetchall(), columns=list(res.keys()))


def explain(label: str, sql: str) -> None:
    """Друкує і зберігає план EXPLAIN (ANALYZE, BUFFERS)."""
    with engine.connect() as conn:
        rows = conn.exec_driver_sql(sql).fetchall()
    plan = '\n'.join(r[0] for r in rows)
    EXPLAINS[label] = plan
    print(f'=== {label} ===')
    print(plan)

### 1.3. Дані NYC TLC Yellow Taxi (Варіант A) з автоматичним fallback (Варіант B)

Джерело: [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), файли `yellow_tripdata_YYYY-MM.parquet` з CloudFront TLC.

Рішення щодо обсягу:

* Беру **дві місячні партиції — лютий і березень 2024** (60 днів). З одним місяцем (31 день) ознака `trips_lag_30d` була б непорожньою лише для останнього дня місяця, і фінальний view (`WHERE trips_lag_30d IS NOT NULL`) майже нічого не повертав би.
* Із кожного місяця беру **випадкову вибірку 250 000 поїздок** (`random_state=42`, разом 500 000 — в межах рекомендованих 300–500 тис.). Не `head()`, бо рядки у файлі згруповані за часом, і перші N рядків покривають лише кілька днів місяця.
* Відкидаю записи з `tpep_pickup_datetime` поза місяцем файлу (у TLC-файлах трапляються «чужі» дати, напр. 2002 або 2009 рік).
* Базове очищення: `0 < fare_amount < 500`, `0 < trip_distance < 100`, `PULocationID IS NOT NULL`.

Якщо завантаження неможливе (немає мережі, змінився URL тощо), notebook **автоматично** переходить на stub-schema (Варіант B), і змінна `DATA_SOURCE` це фіксує. Примусово увімкнути stub можна через `USE_REAL_DATA = False`.

In [ ]:
USE_REAL_DATA = True
MONTHS = ['2024-02', '2024-03']
ROWS_PER_MONTH = 250_000
RANDOM_STATE = 42

BASE_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/'
DATA_DIR = '/content' if os.path.isdir('/content') else './data'
os.makedirs(DATA_DIR, exist_ok=True)

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]


def load_month(ym: str) -> pd.DataFrame:
    fname = f'yellow_tripdata_{ym}.parquet'
    path = os.path.join(DATA_DIR, fname)
    if not os.path.exists(path):
        tmp_path = path + '.part'
        urlretrieve(BASE_URL + fname, tmp_path)
        os.replace(tmp_path, path)          # без «битих» напівзавантажених файлів
    m = pd.read_parquet(path, columns=columns)
    start = pd.Timestamp(f'{ym}-01')
    end = start + pd.offsets.MonthBegin(1)
    m = m[(m['tpep_pickup_datetime'] >= start) & (m['tpep_pickup_datetime'] < end)]
    if len(m) > ROWS_PER_MONTH:
        m = m.sample(n=ROWS_PER_MONTH, random_state=RANDOM_STATE)
    print(f'{fname}: {len(m):,} rows після фільтра за місяцем і вибірки')
    return m


DATA_SOURCE = 'stub'
if USE_REAL_DATA:
    try:
        df = pd.concat([load_month(ym) for ym in MONTHS], ignore_index=True)

        # Basic cleaning before loading to PostgreSQL
        df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
        df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
        df = df[df['PULocationID'].notna()]

        df.to_sql(
            'nyc_taxi_raw',
            engine,
            if_exists='replace',
            index=False,
            chunksize=10_000,
        )
        DATA_SOURCE = 'nyc_tlc'
    except Exception as exc:  # мережа / URL / пам'ять -> stub fallback
        print('Не вдалося завантажити TLC-дані, використовую stub-schema fallback.')
        print('Причина:', repr(exc)[:300])

print('DATA_SOURCE =', DATA_SOURCE)
if DATA_SOURCE == 'nyc_tlc':
    display(q('SELECT COUNT(*) AS n_rows, MIN(tpep_pickup_datetime) AS min_pickup, '
              'MAX(tpep_pickup_datetime) AS max_pickup FROM nyc_taxi_raw'))

### 1.4. Daily aggregation `t7_hw_taxi_daily`

* **Варіант A (TLC):** агрегат `(pickup_zone_id, ts_day)` із `nyc_taxi_raw`. TLC-час — локальний NYC wall-clock, тому день = локальний день, `ts_day` має тип `DATE`.
* **Варіант B (stub):** 60 днів × 5 zones = 300 рядків, дата зафіксована як `DATE '2024-03-31'` (не `CURRENT_DATE`) — результати стабільні між запусками.

Саме ця таблиця використовується в усіх time-series запитах.

In [ ]:
DAILY_FROM_TLC_SQL = """
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER                                   AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE  AS ts_day,
    COUNT(*)::INTEGER                                         AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2)                          AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2)                        AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
  AND tpep_pickup_datetime >= TIMESTAMP '2024-02-01'
  AND tpep_pickup_datetime <  TIMESTAMP '2024-04-01'
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);

ANALYZE t7_hw_taxi_daily;
"""

DAILY_STUB_SQL = """
DROP VIEW IF EXISTS v_t7_hw_taxi_features;
DROP TABLE IF EXISTS t7_hw_taxi_daily;

CREATE TABLE t7_hw_taxi_daily (
    pickup_zone_id INTEGER NOT NULL,
    ts_day         DATE    NOT NULL,
    trip_count     INTEGER NOT NULL CHECK (trip_count >= 0),
    avg_fare       NUMERIC(10, 2) NOT NULL CHECK (avg_fare >= 0),
    total_distance NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),
    PRIMARY KEY (pickup_zone_id, ts_day)
);

INSERT INTO t7_hw_taxi_daily (pickup_zone_id, ts_day, trip_count, avg_fare, total_distance)
SELECT
    z.zone_id,
    DATE '2024-03-31' - (59 - n)                                   AS ts_day,
    500 + (n * 7) + z.zone_id * 50 + (n % 5) * 30                  AS trip_count,
    (12.5 + z.zone_id * 0.7 + (n % 4) * 0.4)::NUMERIC(10, 2)       AS avg_fare,
    (1500 + n * 25 + z.zone_id * 120)::NUMERIC(12, 2)              AS total_distance
FROM generate_series(0, 59) AS n
CROSS JOIN (VALUES (1), (2), (3), (4), (5)) AS z(zone_id);

ANALYZE t7_hw_taxi_daily;
"""

run_sql(DAILY_FROM_TLC_SQL if DATA_SOURCE == 'nyc_tlc' else DAILY_STUB_SQL)

summary = q("""
SELECT COUNT(*)                        AS n_rows,
       COUNT(DISTINCT pickup_zone_id)  AS n_zones,
       MIN(ts_day)                     AS min_day,
       MAX(ts_day)                     AS max_day,
       pg_typeof(MIN(ts_day))::TEXT    AS ts_day_type
FROM t7_hw_taxi_daily
""")
display(summary)
assert summary.loc[0, 'n_rows'] > 100, 't7_hw_taxi_daily має містити понад 100 рядків'

q("""
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10
""")

### 1.5. Часові типи: `TIMESTAMP` (локальний wall-clock) vs `TIMESTAMPTZ` (момент часу)

Для production event-log момент часу варто зберігати як `TIMESTAMPTZ`, інтерпретуючи raw-час TLC через `AT TIME ZONE 'America/New_York'`. Таблиця `t7_hw_taxi_trips_typed` будується лише для Варіанта A (для stub немає поїздкових raw-даних). Демонстрація DST (10.03.2024 о 02:00 у NYC годинник переводили на 03:00) працює в обох варіантах.

In [ ]:
if DATA_SOURCE == 'nyc_tlc':
    run_sql("""
    DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

    CREATE TABLE t7_hw_taxi_trips_typed AS
    SELECT
        "PULocationID"::INTEGER AS pickup_zone_id,
        "DOLocationID"::INTEGER AS dropoff_zone_id,
        (tpep_pickup_datetime::TIMESTAMP  AT TIME ZONE 'America/New_York') AS pickup_at,
        (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
        trip_distance::NUMERIC(10, 2)  AS trip_distance,
        fare_amount::NUMERIC(10, 2)    AS fare_amount,
        passenger_count::INTEGER       AS passenger_count
    FROM nyc_taxi_raw
    WHERE fare_amount > 0
      AND fare_amount < 500
      AND trip_distance > 0
      AND trip_distance < 100;
    """)
    display(q("""
    SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
    FROM t7_hw_taxi_trips_typed
    LIMIT 5
    """))
else:
    print('stub-schema: t7_hw_taxi_trips_typed не створюється (немає raw-поїздок)')

# DST: одна й та сама «локальна» різниця у 2 години не дорівнює 2 годинам реального часу
q("""
SELECT
    TIMESTAMP '2024-03-10 03:30' - TIMESTAMP '2024-03-10 01:30'                       AS naive_diff,
    (TIMESTAMP '2024-03-10 03:30' AT TIME ZONE 'America/New_York')
  - (TIMESTAMP '2024-03-10 01:30' AT TIME ZONE 'America/New_York')                    AS real_elapsed,
    pg_typeof(TIMESTAMP '2024-03-10 01:30' AT TIME ZONE 'America/New_York')::TEXT     AS converted_type
""")

### 1.6. Діагностика пропусків у календарі

`LAG(x, 7)` рахує **7 рядків назад**, а не 7 календарних днів. Якщо в zone є дні без поїздок (рядок відсутній), «lag-7d» тихо стає «lag-7 спостережень». Перевіримо, скільки рядків мають розрив із попереднім рядком тієї ж zone. Rolling-вікна нижче використовують `RANGE ... INTERVAL`, тому до пропусків стійкі; для LAG у production варто будувати calendar-spine (`generate_series` + `LEFT JOIN`).

In [ ]:
q("""
WITH d AS (
    SELECT pickup_zone_id,
           ts_day,
           ts_day - LAG(ts_day) OVER (PARTITION BY pickup_zone_id ORDER BY ts_day) AS gap_days
    FROM t7_hw_taxi_daily
)
SELECT COUNT(DISTINCT pickup_zone_id)                                     AS n_zones,
       COUNT(*)                                                           AS n_rows,
       COUNT(*) FILTER (WHERE gap_days > 1)                               AS rows_after_gap,
       COUNT(DISTINCT pickup_zone_id) FILTER (WHERE gap_days > 1)         AS zones_with_gaps,
       MAX(gap_days)                                                      AS max_gap_days
FROM d
""")

## Завдання 2. Time-series ML features

Усі ознаки рахуються на рівні pickup-zone: кожне вікно містить `PARTITION BY pickup_zone_id`. Жодна ознака не використовує майбутні рядки (`LEAD` / `FOLLOWING` відсутні) — немає data leakage.

### Feature 1. Lag features: lag-1d, lag-7d, lag-30d

In [ ]:
q("""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

На перших 30 рядках кожної zone `trips_lag_30d` = NULL — попередньої історії ще немає. Для training dataset із повною історією такі рядки відфільтровуються (`WHERE trips_lag_30d IS NOT NULL`).

### Feature 2. Rolling 7-day average fare

«7 календарних днів включно з поточним» = `RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW` (не `'7 days'` — це був би off-by-one на 8 днів). `ts_day` приводиться до `TIMESTAMP`, щоб frame `RANGE ... INTERVAL` був однозначним.

In [ ]:
q("""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

#### ROWS vs RANGE на даних із пропусками

Щоб показати різницю, штучно приберемо кілька днів у першій zone (лише в CTE, таблиця не змінюється). `ROWS 6 PRECEDING` бере 7 останніх **рядків** і «розтягується» на 9+ календарних днів, а `RANGE '6 days'` завжди охоплює рівно 7 календарних днів (і менше рядків, якщо днів бракує).

In [ ]:
q("""
WITH first_zone AS (
    SELECT pickup_zone_id AS zid, MIN(ts_day) AS d0
    FROM t7_hw_taxi_daily
    WHERE pickup_zone_id = (SELECT MIN(pickup_zone_id) FROM t7_hw_taxi_daily)
    GROUP BY pickup_zone_id
),
z AS (   -- «видаляємо» 3-й, 4-й і 8-й день від початку історії zone
    SELECT d.*
    FROM t7_hw_taxi_daily AS d
    JOIN first_zone AS f ON d.pickup_zone_id = f.zid
    WHERE d.ts_day NOT IN (f.d0 + 3, f.d0 + 4, f.d0 + 8)
),
w AS (
    SELECT
        pickup_zone_id,
        ts_day,
        COUNT(*)    OVER w_rows  AS rows_frame_n,
        MIN(ts_day) OVER w_rows  AS rows_frame_start,
        COUNT(*)    OVER w_range AS range_frame_n,
        MIN(ts_day) OVER w_range AS range_frame_start
    FROM z
    WINDOW
        w_rows  AS (PARTITION BY pickup_zone_id ORDER BY ts_day
                    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),
        w_range AS (PARTITION BY pickup_zone_id ORDER BY ts_day::TIMESTAMP
                    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW)
)
SELECT *,
       ts_day - rows_frame_start  + 1 AS rows_calendar_span_days,
       ts_day - range_frame_start + 1 AS range_calendar_span_days
FROM w
ORDER BY ts_day
OFFSET 6
LIMIT 8
""")

### Feature 3. Rolling 30-day fare volatility

In [ ]:
q("""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

На першому рядку zone `STDDEV_SAMP` = NULL (вибіркове стандартне відхилення з одного значення не визначене).

### Feature 4. Expanding cumulative trips

In [ ]:
q("""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

### Combined feature view `v_t7_hw_taxi_features`

Усі чотири window definitions (`w_pid`, `w7`, `w30`, `we`) містять `PARTITION BY pickup_zone_id`.

In [ ]:
run_sql("""
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,
        LAG(trip_count, 1)    OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)    OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30)   OVER w_pid AS trips_lag_30d,
        AVG(avg_fare)         OVER w7    AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30   AS rolling_std_fare_30d,
        SUM(trip_count)       OVER we    AS cumulative_trips,
        AVG(trip_count)       OVER we    AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
""")

q("""
SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20
""")

In [ ]:
# Перевірка: кожне вікно у view партиціоноване за pickup_zone_id
viewdef = q("SELECT pg_get_viewdef('v_t7_hw_taxi_features'::regclass, true) AS d").loc[0, 'd']
n_windows = len(re.findall(r'\bAS \(', viewdef.split('WINDOW', 1)[1].split('SELECT', 1)[0]))
n_partitioned = len(re.findall(r'PARTITION BY (?:\w+\.)?pickup_zone_id', viewdef))
print(f'windows у view: {n_windows}, з PARTITION BY pickup_zone_id: {n_partitioned}')
assert n_partitioned == n_windows == 4

q("""
SELECT COUNT(*)                       AS n_feature_rows,
       COUNT(DISTINCT pickup_zone_id) AS n_zones,
       MIN(ts_day)                    AS first_day,
       MAX(ts_day)                    AS last_day
FROM v_t7_hw_taxi_features
""")

#### Чому `PARTITION BY` критичний: тиха ML-помилка без нього

Порівняємо `trips_lag_1d` на **першому дні кожної zone** з партиціонуванням і без нього. Без `PARTITION BY` перший рядок нової zone отримує `trip_count` останнього дня попередньої zone — модель «бачить» історію іншої сутності.

In [ ]:
q("""
WITH lags AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 1) OVER (PARTITION BY pickup_zone_id ORDER BY ts_day)  AS lag_1d_correct,
        LAG(trip_count, 1) OVER (ORDER BY pickup_zone_id, ts_day)              AS lag_1d_leaky,
        ROW_NUMBER()       OVER (PARTITION BY pickup_zone_id ORDER BY ts_day)  AS rn
    FROM t7_hw_taxi_daily
)
SELECT pickup_zone_id, ts_day, trip_count, lag_1d_correct, lag_1d_leaky
FROM lags
WHERE rn = 1
ORDER BY pickup_zone_id
LIMIT 6
""")

## Завдання 3. JSONB-таблиця для подій `t7_hw_events`

50 000 синтетичних подій, 1 000 користувачів, 60 днів, **5 event-типів** із різними схемами payload:

| event_type | ключі payload |
|---|---|
| `page_view` | `url`, `referrer`, `device` |
| `click` | `element_id`, `page_url`, `device` |
| `add_to_cart` | `product_id`, `price`, `quantity`, `tags[]` |
| `checkout` | `order_id`, `total`, `items_count`, `payment_method` |
| `recommendation_shown` | `model_name`, `model_version`, `recommendations[]`, `experiment` |

`event_ts` — `TIMESTAMPTZ` з фіксованою опорною точкою (`2024-04-01 00:00 UTC`), а не `NOW()`, тож дані відтворювані.

In [ ]:
N_EVENTS = 50_000

run_sql(f"""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id   INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id    INTEGER     NOT NULL,
    event_ts   TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type TEXT        NOT NULL,
    payload    JSONB       NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00' - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE 'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url',      '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device',   (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url',   '/category/' || ((n % 20) + 1),
            'device',     (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price',      ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity',   (n % 5) + 1,
            'tags',       jsonb_build_array((ARRAY['promo','new','clearance'])[(n % 3) + 1])
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id',       'o_' || (n + 100000),
            'total',          ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count',    (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name',      'rec_v' || ((n % 5) + 1),
            'model_version',   '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                                   (n % 200) + 1,
                                   ((n + 1) % 200) + 1,
                                   ((n + 2) % 200) + 1
                               ),
            'experiment',      'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, {N_EVENTS}) AS n;

ANALYZE t7_hw_events;
""")

q("""
SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type
""")

In [ ]:
# Різні схеми payload: набір top-level ключів для кожного event_type + розмір таблиці
q("""
SELECT event_type,
       (SELECT string_agg(k, ', ' ORDER BY k) FROM jsonb_object_keys(payload) AS k) AS payload_keys,
       COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY 1, 2
ORDER BY 1
""")

In [ ]:
q("""
SELECT pg_size_pretty(pg_relation_size('t7_hw_events'))        AS heap_size,
       pg_size_pretty(pg_total_relation_size('t7_hw_events'))  AS total_size_with_indexes,
       (SELECT relpages FROM pg_class WHERE relname = 't7_hw_events') AS heap_pages
""")

### JSONB Query 1. Containment `@>`

In [ ]:
q("""
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10
""")

### JSONB Query 2. Key existence `?` та `?|`

`?`, `?|`, `?&` перевіряють лише **top-level** ключі. Для їх індексації через GIN потрібен default `jsonb_ops`, а не `jsonb_path_ops` (див. додатковий експеримент у Завданні 4).

In [ ]:
q("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10
""")

In [ ]:
q("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10
""")

In [ ]:
# Додатково: ?& (усі ключі) та підрахунок, який підтверджує, що ключі живуть лише у «своїх» event_type
q("""
SELECT event_type,
       COUNT(*) FILTER (WHERE payload ? 'tags')                                AS has_tags,
       COUNT(*) FILTER (WHERE payload ?| ARRAY['experiment', 'model_version']) AS has_exp_or_version,
       COUNT(*) FILTER (WHERE payload ?& ARRAY['order_id', 'total'])            AS has_order_and_total
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type
""")

### JSONB Query 3. JSONPath (`jsonb_path_query`, `jsonb_path_exists`)

`jsonb_path_query` — set-returning функція: кожен елемент масиву `recommendations` стає окремим рядком (3 рядки на подію).

In [ ]:
q("""
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15
""")

In [ ]:
q("""
SELECT
    event_id,
    payload ->> 'order_id'                AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10
""")

In [ ]:
# Еквівалент через оператор @? — саме таку форму JSONPath-фільтра може підхопити GIN-індекс
q("""
SELECT COUNT(*) AS n_checkouts_total_gt_300
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @? '$.total ? (@ > 300)'
""")

## Завдання 4. GIN-index demo + EXPLAIN (ANALYZE, BUFFERS)

Чотири обов'язкові плани зберігаються в `EXPLAINS` і зведені в таблицю метрик перед Reflection.

### Step 1. Containment query без GIN-індексу

In [ ]:
run_sql("""
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;
ANALYZE t7_hw_events;
""")

explain('1. @> payment_method=card, без GIN', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

### Step 2. GIN-індекс (`jsonb_path_ops`) для containment / JSONPath workload

In [ ]:
run_sql("""
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);
ANALYZE t7_hw_events;
""")

q("""
SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size,
       pg_size_pretty(pg_relation_size('t7_hw_events'))                       AS heap_size
""")

### Step 3. Той самий containment query після GIN

In [ ]:
explain('2. @> payment_method=card, з GIN jsonb_path_ops', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card')
""")

In [ ]:
explain('2b. @> referrer=google, з GIN jsonb_path_ops', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google')
""")

### Step 4. Коли GIN не допомагає: path-specific text equality

`payload ->> 'device' = 'mobile'` — це рівність **результату виразу** (text), а не JSONB-оператор `@>` / `@?`. GIN-opclass не має стратегії для такого виразу, тому planner його не використає навіть за наявності `idx_t7_events_payload_gin_pathops`.

In [ ]:
explain('3. ->> device = mobile, без expression index (GIN є)', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

Тепер створимо expression B-tree index і повторимо запит.

In [ ]:
run_sql("""
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));
ANALYZE t7_hw_events;
""")

explain('4. ->> device = mobile, з expression B-tree', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile'
""")

In [ ]:
# Для порівняння: та сама умова у формі containment іде через GIN, а не через B-tree
explain('4b. @> device=mobile (containment-форма)', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> '{"device": "mobile"}'::jsonb
""")

### Додатково: `jsonb_ops` vs `jsonb_path_ops` для key existence `?`

`jsonb_path_ops` індексує лише хеші шляхів «ключ→значення», тому не підтримує `?`, `?|`, `?&`. Default `jsonb_ops` індексує ключі й значення окремо — підтримує ширший набір операторів, але більший за розміром. Після експерименту `jsonb_ops`-індекс видаляється, щоб не впливати на подальші плани.

In [ ]:
explain('5a. payload ? tags, лише jsonb_path_ops', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*) FROM t7_hw_events WHERE payload ? 'tags'
""")

run_sql("""
CREATE INDEX idx_t7_events_payload_gin_ops
    ON t7_hw_events USING GIN (payload);       -- default opclass = jsonb_ops
ANALYZE t7_hw_events;
""")

explain('5b. payload ? tags, з GIN jsonb_ops', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*) FROM t7_hw_events WHERE payload ? 'tags'
""")

display(q("""
SELECT indexrelid::regclass::TEXT                      AS index_name,
       pg_size_pretty(pg_relation_size(indexrelid))    AS index_size
FROM pg_index
WHERE indrelid = 't7_hw_events'::regclass
ORDER BY pg_relation_size(indexrelid) DESC
"""))

run_sql("""
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
ANALYZE t7_hw_events;
""")

#### Write-cost GIN: вставка 10 000 подій з індексами і без

Порівнюємо час однакового `INSERT` у дві копії таблиці: без індексів і з тими ж індексами, що на `t7_hw_events` (GIN `jsonb_path_ops` + expression B-tree). Обидві копії тимчасові й видаляються.

In [ ]:
import time

def timed_insert(table: str) -> float:
    t0 = time.perf_counter()
    run_sql(f"""
    INSERT INTO {table} (user_id, event_ts, event_type, payload)
    SELECT user_id, event_ts, event_type, payload
    FROM t7_hw_events
    ORDER BY event_id
    LIMIT 10000;
    """)
    return (time.perf_counter() - t0) * 1000

run_sql("""
DROP TABLE IF EXISTS t7_hw_events_noidx;
DROP TABLE IF EXISTS t7_hw_events_idx;
CREATE TABLE t7_hw_events_noidx (LIKE t7_hw_events INCLUDING DEFAULTS INCLUDING IDENTITY);
CREATE TABLE t7_hw_events_idx   (LIKE t7_hw_events INCLUDING DEFAULTS INCLUDING IDENTITY);
CREATE INDEX ON t7_hw_events_idx USING GIN (payload jsonb_path_ops);
CREATE INDEX ON t7_hw_events_idx ((payload ->> 'device'));
""")

# Прогрів, потім вимірювання (беремо медіану з 3 повторів)
timed_insert('t7_hw_events_noidx'); timed_insert('t7_hw_events_idx')
noidx_ms = sorted(timed_insert('t7_hw_events_noidx') for _ in range(3))[1]
idx_ms   = sorted(timed_insert('t7_hw_events_idx')   for _ in range(3))[1]

write_cost = pd.DataFrame({
    'table': ['без індексів', 'GIN jsonb_path_ops + B-tree(device)'],
    'insert_10k_ms (median of 3)': [round(noidx_ms, 1), round(idx_ms, 1)],
})
write_cost['slowdown_x'] = (write_cost['insert_10k_ms (median of 3)'] / noidx_ms).round(2)
display(write_cost)

run_sql("""
DROP TABLE IF EXISTS t7_hw_events_noidx;
DROP TABLE IF EXISTS t7_hw_events_idx;
""")

## Завдання 5. JSONB vs normalized

### Step 1. Нормалізована таблиця для checkout-подій

Типізовані колонки + `CHECK`-обмеження: невалідний `payment_method` або `items_count <= 0` не потрапить у таблицю (у JSONB таку перевірку довелося б робити окремо).

In [ ]:
run_sql("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id       INTEGER PRIMARY KEY,
    user_id        INTEGER        NOT NULL,
    event_ts       TIMESTAMPTZ    NOT NULL,
    order_id       TEXT           NOT NULL,
    total          NUMERIC(10, 2) NOT NULL,
    items_count    INTEGER        NOT NULL CHECK (items_count > 0),
    payment_method TEXT           NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id, user_id, event_ts, order_id, total, items_count, payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id'                 AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2)  AS total,
    (payload ->> 'items_count')::INTEGER   AS items_count,
    payload ->> 'payment_method'           AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;
""")

q("""
SELECT COUNT(*) AS n_checkouts,
       pg_size_pretty(pg_relation_size('t7_hw_checkouts_norm')) AS heap_size,
       (SELECT relpages FROM pg_class WHERE relname = 't7_hw_checkouts_norm') AS heap_pages
FROM t7_hw_checkouts_norm
""")

### Step 2. Один аналітичний запит двома способами

Спершу переконаємося, що обидві версії повертають однаковий результат, потім порівняємо плани.

In [ ]:
jsonb_res = q("""
SELECT payload ->> 'payment_method' AS payment_method,
       ROUND(AVG((payload ->> 'total')::NUMERIC), 2) AS avg_total,
       COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3
""")
norm_res = q("""
SELECT payment_method,
       ROUND(AVG(total), 2) AS avg_total,
       COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
""")
display(jsonb_res)
print('Результати однакові:', jsonb_res.equals(norm_res))

In [ ]:
explain('6. JSONB: avg total by payment_method', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3
""")

In [ ]:
explain('7. Normalized: avg total by payment_method', """
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3
""")

### Step 3. Bonus: MongoDB-equivalent (pseudocode, не виконується)

```javascript
// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode — не виконується у PostgreSQL
db.events.createIndex({ event_type: 1, "payload.payment_method": 1 });

db.events.aggregate([
  { $match: { event_type: "checkout" } },
  { $group: {
      _id: "$payload.payment_method",
      avg_total: { $avg: { $toDouble: "$payload.total" } },
      n: { $sum: 1 }
  }},
  { $sort: { avg_total: -1 } },
  { $limit: 3 }
]);

// Аналог containment-запиту payload @> {"payment_method": "card"}:
db.events.countDocuments({ "payload.payment_method": "card" });
```

**Різниця підходів:**

* **PostgreSQL JSONB** зручний, коли поруч потрібні SQL, `JOIN` із нормалізованими таблицями (users, orders, feature tables), ACID-транзакції та одна система для OLTP і аналітики. Ціна — `->>` + каст на кожному рядку і відсутність статистики по ключах JSONB без expression index.
* **MongoDB aggregation pipeline** природний для document-first workloads: документ = одиниця читання/запису, схема еволюціонує без міграцій, горизонтальне шардування вбудоване. Але `JOIN` (`$lookup`) і багатодокументні транзакції там дорожчі й менш звичні.
* **High-volume raw clickstream** (мільйони подій/год, append-only, рідкі точкові читання) часто краще жити в document store / data lake (Parquet + Spark/DuckDB) — дешеве зберігання, колонкове читання, без write-amplification від GIN.
* **Curated ML features** (стабільний набір колонок, типи, constraints, point-in-time joins) зручніше тримати у typed SQL-таблицях — як `t7_hw_checkouts_norm` або `v_t7_hw_taxi_features`.

## Зведення метрик EXPLAIN (для Reflection)

Таблиця будується автоматично з планів, збережених у `EXPLAINS`, тож вона завжди відповідає поточному запуску notebook.

In [ ]:
NODE_RE = re.compile(r'(Parallel Seq Scan|Seq Scan|Bitmap Heap Scan|Bitmap Index Scan|'
                     r'Index Only Scan|Index Scan|HashAggregate|GroupAggregate)')


def plan_metrics(plan: str) -> dict:
    first = plan.splitlines()[0]
    cost = re.search(r'cost=[\d.]+\.\.([\d.]+)', first)
    nodes = []
    for n in NODE_RE.findall(plan):
        if n not in nodes:
            nodes.append(n)
    scan = re.search(r'Scan on [^\n]*?rows=(\d+) width[^\n]*?actual time=[\d.]+\.\.[\d.]+ rows=(\d+)', plan)
    buffers = re.search(r'Buffers: shared ([^\n]+)', plan)
    exec_ms = re.search(r'Execution Time: ([\d.]+) ms', plan)
    return {
        'total_cost': float(cost.group(1)) if cost else None,
        'execution_ms': float(exec_ms.group(1)) if exec_ms else None,
        'plan_nodes': ' → '.join(nodes),
        'scan_rows_est/actual': f'{scan.group(1)}/{scan.group(2)}' if scan else None,
        'top_buffers': buffers.group(1) if buffers else None,
    }


metrics = pd.DataFrame([{'query': k, **plan_metrics(v)} for k, v in EXPLAINS.items()])
metrics

## Reflection

> Цифри нижче взято з мого запуску (таблиця «Зведення метрик EXPLAIN» вище). Мілісекунди між запусками коливаються, але оператори плану, `rows` і `Buffers` на детермінованому event-log відтворюються.

**1. До і після GIN.** Без індексу `@> {"payment_method":"card"}` виконувався як **Seq Scan** (total cost ≈ 1760, ≈ 35 ms): прочитано всі 1000 сторінок, `Rows Removed by Filter: 46 666`. Після `GIN jsonb_path_ops` план змінився на **Bitmap Index Scan → Bitmap Heap Scan**: cost ≈ 1085, ≈ 5–8 ms. Сам Bitmap Index Scan торкнувся лише 4 сторінок індексу і за ≈ 0.4 ms знайшов 3 334 рядки. При цьому `Heap Blocks: exact=1000`: checkout-події рівномірно розкидані по таблиці, тому прочитати довелося все одно всі сторінки. Виграш дав не меншим I/O, а тим, що containment перевірявся на 3 334 рядках замість 50 000.

**2. Чому GIN не для `->> 'device' = 'mobile'`.** Це порівняння тексту, отриманого з виразу, а не JSONB-оператор. Жоден GIN-opclass такого виразу не підтримує, тож навіть за наявності GIN план лишився **Seq Scan** (≈ 9–14 ms). Ще одна проблема: без статистики planner оцінив `rows=250` при фактичних 8 333. Expression B-tree дав **Bitmap Index Scan** на `idx_t7_events_device_btree` (≈ 3 ms), а `ANALYZE` зібрав статистику по виразу, і оцінка стала `rows≈8 377`.

**3. Read/write trade-off.** Читання `@>`/`@?` прискорюється приблизно в 5–7 разів. Ціна така: GIN `jsonb_path_ops` займає 1144 kB (≈ 14 % від heap у 8000 kB), а `jsonb_ops` — 1696 kB. Вставка 10 000 подій із GIN + B-tree зайняла ≈ 99 ms проти ≈ 52 ms без індексів (у ≈ 1.9 раза повільніше). До цього додаються витрати на VACUUM і pending list.

**4. JSONB чи normalized.** Обидві версії повертають той самий результат. Але JSONB-версія прочитала **1000 буферів** (cost ≈ 1960, ≈ 15 ms), бо пройшла всі 50 000 подій і для кожної робила `->>` і каст. Normalized прочитала **97 буферів** (cost ≈ 269, ≈ 3 ms). JSONB я залишив би для сирого event-log зі схемами, що змінюються, і рідкісних запитів. Для curated-ознак, які щодня читають training і serving, краще підходять типізовані таблиці з `CHECK`.

**5. Новий event-type `refund`.** У JSONB достатньо додати `INSERT ... ('refund', '{"order_id":…, "amount":…, "reason":…}')`: DDL не потрібен, наявний GIN покриває нові ключі. Валідацію (наприклад, `CHECK` через `jsonb_path_exists`) і обробку в споживачах треба дописати окремо. У normalized-підході потрібна міграція: `CREATE TABLE t7_hw_refunds_norm` з FK на `order_id`, обмеженнями й індексами, оновлення ETL і backfill історичних подій.

## Фінальна перевірка чекліста

In [ ]:
checks = {
    'pgserver / PostgreSQL підключено':                   q('SELECT 1 AS ok').loc[0, 'ok'] == 1,
    't7_hw_taxi_daily > 100 рядків':                     q('SELECT COUNT(*) AS n FROM t7_hw_taxi_daily').loc[0, 'n'] > 100,
    'view з lag/rolling/expanding не порожній':          q('SELECT COUNT(*) AS n FROM v_t7_hw_taxi_features').loc[0, 'n'] > 0,
    't7_hw_events має >= 5 event-типів':                 q('SELECT COUNT(DISTINCT event_type) AS n FROM t7_hw_events').loc[0, 'n'] >= 5,
    'EXPLAIN до/після GIN збережено':                    all(k in EXPLAINS for k in [
                                                            '1. @> payment_method=card, без GIN',
                                                            '2. @> payment_method=card, з GIN jsonb_path_ops']),
    'EXPLAIN до/після expression index збережено':       all(k in EXPLAINS for k in [
                                                            '3. ->> device = mobile, без expression index (GIN є)',
                                                            '4. ->> device = mobile, з expression B-tree']),
    't7_hw_checkouts_norm створено':                     q('SELECT COUNT(*) AS n FROM t7_hw_checkouts_norm').loc[0, 'n'] > 0,
}
print('DATA_SOURCE =', DATA_SOURCE)
for name, ok in checks.items():
    print(('✅' if ok else '❌'), name)
assert all(checks.values())